# T09 - 利率预测

## 项目概述

利率预测项目使用多模型融合的方法，从多个维度（行情集中度、交易拥挤度、资金供需、银行负债成本、基本面）预测债券收益率的走势和拐点。

### 核心功能
- **子模型1**: 基于行情集中度 - 短期预测（1-3个月）
- **子模型2**: 基于交易拥挤度 - 超短期预测（1个月）
- **子模型3**: 基于资金供需 - 中长期预测（1-2年）
- **子模型4**: 基于银行负债成本 - 中期预测（1-2年）
- **子模型5**: 基于基本面和自然利率 - 长期预测（3-12个月）

### 数据源
- MySQL数据库: bond库、yq库、edb库
- 主要表: marketinfo_curve, market_concentration_90pct, dealtinfo等

### 输出结果
- 各期限收益率预测值
- 拐点识别信号
- 可视化图表

---
## 1. 环境配置

In [ ]:
# 标准库
import os
import sys
import warnings
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta
from io import StringIO

warnings.filterwarnings('ignore')

# 数据处理
import numpy as np
import pandas as pd
from scipy import interpolate, signal
from scipy.stats import invgamma, norm, beta, gamma
from scipy.optimize import minimize
from scipy.linalg import solve, cholesky

# 数据库
import sqlalchemy
from sqlalchemy import create_engine, text
import sqlalchemy.pool

# 机器学习
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
import lightgbm as lgb
from prophet import Prophet

# 可视化
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'WenQuanYi Micro Hei', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False

# 尝试导入plotly
try:
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False

print("环境配置完成")

---
## 2. 配置参数

In [ ]:
# 从环境变量或config模块导入配置
try:
    from config import DB_CONFIG, MODEL_CONFIG, PATH_CONFIG
except ImportError:
    # 默认配置（仅用于演示，实际使用请配置config.py）
    DB_CONFIG = {
        'user': os.environ.get('DB_USER', 'your_user'),
        'password': os.environ.get('DB_PASSWORD', 'your_password'),
        'host': os.environ.get('DB_HOST', 'localhost'),
        'port': os.environ.get('DB_PORT', '3306'),
        'db': 'bond'
    }
    
    MODEL_CONFIG = {
        'model1': {'type': 'short_term', 'horizon': 63, 'weights': {'sentiment': 0.4, 'concentration': 0.6}},
        'model2': {'type': 'ultra_short', 'horizon': 30, 'weights': {'trading_heat': 1.0}},
        'model3': {'type': 'medium_long', 'horizon': 365, 'weights': {'supply_demand': 1.0}},
        'model4': {'type': 'medium', 'horizon': 365, 'weights': {'bank_cost': 1.0}},
        'model5': {'type': 'long', 'horizon': 360, 'weights': {'fundamental': 0.5, 'natural_rate': 0.5}}
    }
    
    PATH_CONFIG = {
        'output_dir': './output',
        'plot_dir': './assets'
    }

print("配置参数加载完成")

In [ ]:
def create_db_engine(db_name='bond'):
    """创建数据库连接引擎"""
    config = DB_CONFIG.copy()
    config['db'] = db_name
    connection_string = f"mysql+pymysql://{config['user']}:{config['password']}@{config['host']}:{config['port']}/{config['db']}"
    return create_engine(connection_string)

# 创建各数据库引擎
engine_bond = create_db_engine('bond')
engine_yq = create_db_engine('yq')
engine_edb = create_db_engine('edb')

print("数据库连接创建完成")

---
## 3. 数据获取

### 3.1 行情集中度数据获取

In [ ]:
def get_concentration_data(engine, start_date, end_date):
    """获取市场行情集中度数据"""
    sql = f"""
    SELECT `dt`, `concentration_90pct`
    FROM bond.market_concentration_90pct
    WHERE dt BETWEEN '{start_date}' AND '{end_date}'
    ORDER BY `dt`
    """
    df = pd.read_sql(sql, engine)
    df.rename(columns={'concentration_90pct': 'concentration'}, inplace=True)
    df['dt'] = pd.to_datetime(df['dt'])
    return df

# 示例使用
# end_date = datetime.now().strftime('%Y-%m-%d')
# start_date = (datetime.now() - relativedelta(years=5)).strftime('%Y-%m-%d')
# df_concentration = get_concentration_data(engine_bond, start_date, end_date)
# df_concentration.head()

### 3.2 国债收益率数据获取

In [ ]:
def get_yield_data(engine, start_date, end_date):
    """获取国债平均到期收益率"""
    sql = f"""
    SELECT 
        A.dt as trade_date,
        avg(A.CLOSE) AS avg_yield
    FROM bond.marketinfo_curve A
    INNER JOIN bond.basicinfo_curve B ON A.trade_code = B.TRADE_CODE
    WHERE A.dt BETWEEN '{start_date}' AND '{end_date}'
      AND B.classify2 = '中债国债到期收益率'
    GROUP BY A.dt
    """
    df = pd.read_sql(sql, engine)
    df['trade_date'] = pd.to_datetime(df['trade_date'])
    return df

def get_10y_yield_data(engine, start_date, end_date):
    """获取10年期国债收益率"""
    sql = f"""
    SELECT `dt`, `close` AS `yield`
    FROM bond.marketinfo_curve
    WHERE trade_code = 'L001619604'
      AND dt BETWEEN '{start_date}' AND '{end_date}'
    ORDER BY `dt`
    """
    df = pd.read_sql(sql, engine)
    df['dt'] = pd.to_datetime(df['dt'])
    return df

### 3.3 交易拥挤度数据获取

In [ ]:
def get_congestion_data(engine, start_date, end_date):
    """计算债券市场的交易拥挤度（活跃券成交占比）"""
    sql = f"""
    SELECT
        DT as trade_date,
        TRADE_CODE as bond_code,
        sum(transaction_amount) / 10000 as turnover
    FROM bond.dealtinfo
    WHERE SEC_NAME LIKE '%%国债%%'
      AND DT BETWEEN '{start_date}' AND '{end_date}'
    GROUP BY trade_date, bond_code
    """
    df = pd.read_sql(sql, engine)
    if df.empty:
        return pd.DataFrame(columns=['trade_date', 'congestion_rate'])
    
    df['trade_date'] = pd.to_datetime(df['trade_date'])
    
    # 计算每日每只债券的成交金额排名
    df['daily_rank'] = df.groupby('trade_date')['turnover'].rank(ascending=False)
    
    # 识别活跃券
    df['is_active'] = (
        (df['daily_rank'] <= 4) &
        (df['turnover'] > df.groupby('trade_date')['turnover'].transform('mean') * 3)
    )

    # 计算活跃券成交占比
    daily_stats = df.groupby(['trade_date', 'is_active'])['turnover'].sum().unstack(fill_value=0)
    if True not in daily_stats.columns:
        daily_stats[True] = 0
    if False not in daily_stats.columns:
        daily_stats[False] = 0

    daily_stats.columns = ['inactive_turnover', 'active_turnover']
    total_turnover = daily_stats['active_turnover'] + daily_stats['inactive_turnover']
    daily_stats['congestion_rate'] = np.where(total_turnover > 0, 
                                               daily_stats['active_turnover'] / total_turnover, 0)
    
    # 使用20日移动平均值作为最终拥挤度指标
    daily_stats.sort_index(inplace=True)
    daily_stats['congestion_rate'] = daily_stats['congestion_rate'].rolling(20, min_periods=1).mean().round(4)
    
    return daily_stats.reset_index()[['trade_date', 'congestion_rate']]

### 3.4 资金供需数据获取

In [ ]:
def get_demand_supply_data(engine, start_date, end_date):
    """获取债市的需求-供给数据"""
    # 计算总需求
    sql_demand = f"""
    SELECT dt, B.SEC_NAME, avg(A.close/10000) as close
    FROM edb.edbdata_ths A
    INNER JOIN edb.edbinfo_ths B ON A.TRADE_CODE=B.TRADE_CODE
    WHERE A.TRADE_CODE in ('S003138026','S005890488','S004054283','S004054284','M004030789','M004030786')
      AND A.dt BETWEEN '{start_date}' AND '{end_date}'
    GROUP BY dt, B.SEC_NAME
    """
    df_demand = pd.read_sql(sql_demand, engine)
    # ... (简化版本，完整实现见源代码)
    
    # 返回需求供给差值
    return pd.DataFrame(columns=['trade_date', 'demand_supply_gap'])

### 3.5 银行负债成本数据获取

In [ ]:
def get_liability_cost_data(engine, start_date, end_date):
    """获取银行负债成本相关数据"""
    sql_templates = {
        'wealth_management': f"""
            SELECT DT AS x, avg(近1年净值年化增长率) AS y 
            FROM yq.理财业绩跟踪
            WHERE 近1年净值年化增长率 is not null 
              AND DT BETWEEN '{start_date}' AND '{end_date}'
            GROUP BY dt ORDER BY dt
        """,
        'deposit_3y': f"""
            SELECT DT AS x, close AS y 
            FROM edb.edbdata 
            WHERE trade_code='M0329279' 
              AND DT BETWEEN '{start_date}' AND '{end_date}'
        """,
        'ncd_rate': f"""
            SELECT dt as x, avg(收益率) as y 
            FROM (
                SELECT A.dt, A.CLOSE AS 收益率
                FROM bond.marketinfo_curve A
                INNER JOIN bond.basicinfo_curve B ON A.trade_code = B.TRADE_CODE
                WHERE A.dt BETWEEN '{start_date}' AND '{end_date}'
                  AND B.classify2 LIKE '中债商业银行同业存单到期收益率%%'
            ) sq
            GROUP BY dt
        """
    }
    
    datasets = {}
    for name, sql in sql_templates.items():
        try:
            datasets[name] = pd.read_sql(sql, engine)
            datasets[name]['x'] = pd.to_datetime(datasets[name]['x'])
        except Exception as e:
            print(f"获取{name}数据失败: {e}")
            datasets[name] = pd.DataFrame(columns=['x', 'y'])
    
    return datasets

---
## 4. 数据处理

In [ ]:
def resample_daily(df, date_col='x', value_col='y'):
    """将数据重采样为日度数据"""
    if df.empty or date_col not in df.columns:
        return pd.DataFrame(columns=[date_col, value_col])
    df = df.set_index(date_col)
    df = df.resample('D').mean()
    df = df.ffill().bfill()
    return df.reset_index()

def merge_data_on_date(dfs, date_col='dt'):
    """按日期合并多个DataFrame"""
    if not dfs:
        return pd.DataFrame()
    
    result = dfs[0]
    for df in dfs[1:]:
        result = pd.merge(result, df, on=date_col, how='outer')
    
    result.sort_values(date_col, inplace=True)
    return result

---
## 5. 核心逻辑

### 5.1 子模型1: 基于行情集中度（短期预测）

In [ ]:
class ConcentrationModel:
    """基于行情集中度的短期预测模型"""
    
    def __init__(self, prediction_days=63):
        self.prediction_days = prediction_days
        self.model = None
        self.features = ['concentration', 'concentration_rol_avg_21d', 
                         'concentration_chg_5d', 'concentration_detrend']
    
    def prepare_features(self, df):
        """准备特征"""
        df = df.copy()
        df['concentration_rol_avg_21d'] = df['concentration'].rolling(window=21).mean()
        df['concentration_chg_5d'] = df['concentration'].diff(5)
        df['concentration_detrend'] = df['concentration'] - df['concentration_rol_avg_21d']
        df['yield_change'] = df['yield'].shift(-self.prediction_days) - df['yield']
        return df
    
    def train(self, df):
        """训练模型"""
        df_model = df.dropna(subset=self.features + ['yield_change'])
        X = df_model[self.features]
        y = df_model['yield_change']
        
        self.model = GradientBoostingRegressor(
            n_estimators=100, 
            learning_rate=0.1, 
            max_depth=3, 
            random_state=42
        )
        self.model.fit(X, y)
        
        print(f"模型R-squared: {self.model.score(X, y):.4f}")
        print("特征重要性:")
        for feature, importance in zip(self.features, self.model.feature_importances_):
            print(f"  - {feature}: {importance:.4f}")
        
        return self
    
    def predict(self, latest_data):
        """执行预测"""
        if self.model is None:
            raise ValueError("模型尚未训练")
        
        latest_features = latest_data[self.features]
        predicted_change = self.model.predict(latest_features)[0]
        latest_yield = latest_data['yield'].iloc[0]
        
        return {
            'predicted_change': predicted_change,
            'current_yield': latest_yield,
            'predicted_yield': latest_yield + predicted_change
        }

### 5.2 子模型2: 基于交易拥挤度（超短期预测）

In [ ]:
class CongestionModel:
    """基于交易拥挤度的超短期预测模型"""
    
    def __init__(self, prediction_days=30):
        self.prediction_days = prediction_days
        self.model = None
        self.features = ['avg_yield', 'congestion_rate', 'congestion_momentum', 'yield_volatility_20d']
    
    def prepare_features(self, df):
        """准备特征"""
        df = df.copy()
        df['congestion_momentum'] = df['congestion_rate'].diff(5)
        df['yield_volatility_20d'] = df['avg_yield'].rolling(20).std()
        df['yield_change'] = df['avg_yield'].shift(-self.prediction_days) - df['avg_yield']
        return df
    
    def train(self, df):
        """训练模型"""
        df_model = df.dropna(subset=self.features + ['yield_change'])
        X = df_model[self.features]
        y = df_model['yield_change']
        
        self.model = lgb.LGBMRegressor(random_state=42)
        self.model.fit(X, y)
        print("LightGBM模型训练完成")
        
        return self
    
    def predict(self, latest_data):
        """执行预测"""
        if self.model is None:
            raise ValueError("模型尚未训练")
        
        latest_features = latest_data[self.features]
        predicted_change = self.model.predict(latest_features)[0]
        latest_yield = latest_data['avg_yield'].iloc[0]
        
        return {
            'predicted_change': predicted_change,
            'current_yield': latest_yield,
            'predicted_yield': latest_yield + predicted_change
        }

### 5.3 子模型3: 基于资金供需（中长期预测）

In [ ]:
class DemandSupplyModel:
    """基于资金供需的中长期预测模型"""
    
    def __init__(self, prediction_days=365):
        self.prediction_days = prediction_days
        self.model_downward = None
        self.model_oscillating = None
        self.features = ['avg_yield', 'demand_supply_gap', 'sin_day', 'cos_day']
    
    def prepare_features(self, df):
        """准备特征，包含市场状态识别"""
        df = df.copy()
        
        # 定义市场状态
        df['gap_trend'] = df['demand_supply_gap'].rolling(90, min_periods=30).mean().diff()
        df['gap_high_threshold'] = df['demand_supply_gap'].rolling(365, min_periods=90).quantile(0.75)
        df['regime_is_downward'] = (df['gap_trend'] > 0) | (df['demand_supply_gap'] > df['gap_high_threshold'])
        
        # 时间特征
        df['yield_t_plus_1'] = df['avg_yield'].shift(-1)
        df['day_of_year'] = df.index.dayofyear if hasattr(df.index, 'dayofyear') else range(len(df))
        df['sin_day'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
        df['cos_day'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)
        
        return df
    
    def train(self, df):
        """训练双模型（单边下行/震荡）"""
        df_model = df.dropna()
        
        df_downward = df_model[df_model['regime_is_downward'] == True]
        df_oscillating = df_model[df_model['regime_is_downward'] == False]
        
        # 训练下行市场模型
        self.model_downward = make_pipeline(
            PolynomialFeatures(degree=2, include_bias=False), 
            LinearRegression()
        )
        self.model_downward.fit(df_downward[self.features], df_downward['yield_t_plus_1'])
        
        # 训练震荡市场模型
        self.model_oscillating = make_pipeline(
            PolynomialFeatures(degree=2, include_bias=False), 
            LinearRegression()
        )
        self.model_oscillating.fit(df_oscillating[self.features], df_oscillating['yield_t_plus_1'])
        
        print("双模型训练完成")
        return self
    
    def predict_path(self, latest_data, days=365):
        """递归预测未来路径"""
        # 判断当前市场状态
        is_downward = latest_data['regime_is_downward'].iloc[-1]
        model = self.model_downward if is_downward else self.model_oscillating
        
        predictions = []
        current_yield = latest_data['avg_yield'].iloc[-1]
        current_gap = latest_data['demand_supply_gap'].iloc[-1]
        
        for i in range(days):
            current_date = latest_data.index[-1] + timedelta(days=i+1)
            day_of_year = current_date.dayofyear
            sin_day = np.sin(2 * np.pi * day_of_year / 365.25)
            cos_day = np.cos(2 * np.pi * day_of_year / 365.25)
            
            features = np.array([[current_yield, current_gap, sin_day, cos_day]])
            predicted_yield = model.predict(features)[0]
            predictions.append({'date': current_date, 'yield': predicted_yield})
            current_yield = predicted_yield
        
        return pd.DataFrame(predictions)

### 5.4 子模型4: 基于银行负债成本（中期预测）

In [ ]:
class LiabilityCostPredictor:
    """银行负债成本预测器"""
    
    def __init__(self, engine):
        self.engine = engine
        self.weights = {
            'wealth_management': 0.30,
            'deposit_3y': 0.25,
            'deposit_6m': 0.25,
            'ncd': 0.20
        }
    
    def calculate_liability_cost(self, df):
        """计算综合负债成本"""
        return (
            df['wealth_management'] * self.weights['wealth_management'] +
            df['deposit_3y'] * self.weights['deposit_3y'] +
            df['deposit_6m'] * self.weights['deposit_6m'] +
            df['ncd'] * self.weights['ncd']
        )
    
    def generate_future_assumptions(self, hist_df, name):
        """使用Prophet模型生成未来预测假设"""
        prophet_df = hist_df.rename(columns={'x': 'ds', 'y': 'y'})
        
        if len(prophet_df) < 2:
            return {}
        
        model = Prophet()
        model.fit(prophet_df)
        
        future = model.make_future_dataframe(periods=365*2)
        forecast = model.predict(future)
        
        key_dates = ['2025-06-30', '2025-12-31', '2026-06-30', '2026-12-31']
        assumptions = {}
        for dt_str in key_dates:
            target_date = pd.to_datetime(dt_str)
            closest = forecast.iloc[forecast['ds'].sub(target_date).abs().idxmin()]
            assumptions[dt_str] = closest['yhat']
        
        return assumptions
    
    def predict(self, prediction_start_date):
        """执行预测"""
        # 获取历史数据并预测
        # 完整实现见源代码 run_prediction.py
        print(f"从 {prediction_start_date} 开始预测")
        return pd.DataFrame()

### 5.5 子模型5: 基于基本面和自然利率（长期预测）

In [ ]:
class NaturalRateModel:
    """基于状态空间模型的自然利率估算"""
    
    def __init__(self):
        self.n_states = 6  # [g_t, z_t, c_t, pi_t, pi_e_t, r_t]
        self.n_obs = 4
        self.params = {}
        self.results = {}
    
    def setup_priors(self):
        """设置参数先验分布"""
        self.params = {
            'alpha_c3': 0.25,
            'alpha_pi': 0.2,
            'gamma_r': 0.8,
            'zeta': 0.3,
            'sigma_g': 0.003,
            'sigma_z': 0.002,
            'sigma_c': 0.008,
            'sigma_pi': 0.005,
            'sigma_r': 0.004
        }
    
    def state_transition(self, state_t_1, params):
        """状态转移方程"""
        g_t_1, z_t_1, c_t_1, pi_t_1, pi_e_t_1, r_t_1 = state_t_1
        
        # 潜在增长率
        g_t = g_t_1
        
        # 自然利率其他因素
        z_t = z_t_1
        
        # 产出缺口
        r_star_t_1 = params['zeta'] * g_t_1 + z_t_1
        r_gap = r_t_1 - r_star_t_1
        c_t = 0.7 * c_t_1 - params['alpha_c3'] * r_gap
        
        # 通胀
        pi_t = pi_e_t_1/4 + params['alpha_pi'] * c_t_1
        
        # 通胀预期
        pi_e_t = 0.9 * pi_e_t_1 + 0.1 * pi_t_1 * 4
        
        # 政策利率
        r_star_t = params['zeta'] * g_t + z_t
        r_t = params['gamma_r'] * r_t_1 + (1 - params['gamma_r']) * r_star_t
        
        return np.array([g_t, z_t, c_t, pi_t, pi_e_t, r_t])
    
    def kalman_filter_smooth(self, obs_data, T):
        """卡尔曼滤波与平滑"""
        # 简化版本，完整实现见 naturalratecn.py
        states = np.zeros((T, self.n_states))
        return states
    
    def estimate(self, data):
        """执行估计"""
        self.setup_priors()
        # 完整实现需要数据准备和优化过程
        print("自然利率模型估计完成")
        return self.results

---
## 6. 执行与测试

In [ ]:
def run_all_models(engine_bond, engine_yq, engine_edb, end_date=None):
    """运行所有预测模型"""
    if end_date is None:
        end_date = datetime.now()
    
    start_date = end_date - relativedelta(years=5)
    start_date_str = start_date.strftime('%Y-%m-%d')
    end_date_str = end_date.strftime('%Y-%m-%d')
    
    results = {}
    
    # 模型1: 行情集中度
    print("\n=== 子模型1: 行情集中度 ===")
    try:
        df_concentration = get_concentration_data(engine_bond, start_date_str, end_date_str)
        df_yield_10y = get_10y_yield_data(engine_bond, start_date_str, end_date_str)
        
        df_merged = pd.merge(df_yield_10y, df_concentration, on='dt', how='inner')
        df_merged.set_index('dt', inplace=True)
        df_merged.dropna(inplace=True)
        
        if not df_merged.empty:
            model1 = ConcentrationModel(prediction_days=63)
            df_features = model1.prepare_features(df_merged)
            model1.train(df_features)
            
            latest_data = df_features.iloc[-1:]
            results['model1'] = model1.predict(latest_data)
            print(f"预测结果: {results['model1']}")
    except Exception as e:
        print(f"模型1执行失败: {e}")
        results['model1'] = None
    
    # 模型2: 交易拥挤度
    print("\n=== 子模型2: 交易拥挤度 ===")
    try:
        df_congestion = get_congestion_data(engine_bond, start_date_str, end_date_str)
        df_yield_avg = get_yield_data(engine_bond, start_date_str, end_date_str)
        
        df_merged = pd.merge(df_yield_avg, df_congestion, on='trade_date', how='inner')
        df_merged.set_index('trade_date', inplace=True)
        df_merged.dropna(inplace=True)
        
        if not df_merged.empty:
            model2 = CongestionModel(prediction_days=30)
            df_features = model2.prepare_features(df_merged)
            model2.train(df_features)
            
            latest_data = df_features.iloc[-1:]
            results['model2'] = model2.predict(latest_data)
            print(f"预测结果: {results['model2']}")
    except Exception as e:
        print(f"模型2执行失败: {e}")
        results['model2'] = None
    
    return results

# 运行测试
# results = run_all_models(engine_bond, engine_yq, engine_edb)

---
## 7. 可视化

In [ ]:
def plot_prediction_results(df_history, df_prediction, title='利率预测结果'):
    """绘制预测结果"""
    fig, ax = plt.subplots(figsize=(14, 7))
    
    # 历史数据
    ax.plot(df_history.index, df_history['yield'], 
            label='历史收益率', color='blue', linewidth=2)
    
    # 预测数据
    if df_prediction is not None and not df_prediction.empty:
        ax.plot(df_prediction['date'], df_prediction['yield'],
                label='预测收益率', color='red', linestyle='--', linewidth=2)
    
    ax.set_xlabel('日期', fontsize=12)
    ax.set_ylabel('收益率', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

def plot_model_comparison(results):
    """绘制多模型预测对比"""
    fig, ax = plt.subplots(figsize=(10, 6))
    
    models = []
    predictions = []
    
    for model_name, result in results.items():
        if result is not None:
            models.append(model_name)
            predictions.append(result.get('predicted_yield', 0))
    
    ax.bar(models, predictions, color=['blue', 'green', 'orange', 'red', 'purple'][:len(models)])
    ax.set_ylabel('预测收益率')
    ax.set_title('各模型预测结果对比')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    return fig

---
## 8. 工具函数

In [ ]:
def save_predictions_to_db(df, engine, table_name):
    """保存预测结果到数据库"""
    try:
        df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=1000)
        print(f"成功保存 {len(df)} 条记录到表 {table_name}")
    except Exception as e:
        print(f"保存失败: {e}")

def calculate_model_metrics(y_true, y_pred):
    """计算模型评估指标"""
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    return {
        'MSE': mse,
        'MAE': mae,
        'R2': r2,
        'RMSE': np.sqrt(mse)
    }

def create_interpolation_curve(key_points, start_date, end_date):
    """创建样条插值曲线"""
    dates = list(key_points.keys())
    values = list(key_points.values())
    
    dates = pd.to_datetime(dates)
    pred_dates = pd.date_range(start=start_date, end=end_date, freq='D')
    
    days = [(d - dates[0]).days for d in dates]
    pred_days = [(d - dates[0]).days for d in pred_dates]
    
    f = interpolate.PchipInterpolator(days, values, extrapolate=True)
    pred_values = f(pred_days)
    
    return pd.DataFrame({'x': pred_dates, 'y': pred_values})

print("工具函数定义完成")

---
## 总结

本Notebook实现了利率预测的多模型框架，包括:

1. **子模型1 - 行情集中度**: 使用GradientBoosting进行短期预测
2. **子模型2 - 交易拥挤度**: 使用LightGBM进行超短期预测
3. **子模型3 - 资金供需**: 使用多项式回归进行中长期预测
4. **子模型4 - 银行负债成本**: 使用Prophet进行中期预测
5. **子模型5 - 自然利率**: 使用卡尔曼滤波进行长期预测

### 使用说明

1. 配置 `config.py` 文件中的数据库连接信息
2. 运行各个子模型进行预测
3. 融合多模型结果进行综合分析

### 注意事项

- 数据库连接需要正确的权限配置
- 各模型对数据量和数据质量有不同要求
- 预测结果仅供参考，不构成投资建议